In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
from pathlib import Path
from itertools import groupby
from operator import itemgetter
from CustomFunctions import DetailedBalance, utils

In [2]:
####### load common directories FOR APPLIED CONFOCAL PCA
time_interval = 5 #sec/frame
whichpcs = [1,2]
basedir = Path('E:/Aaron/Combined_37C_Confocal_PCA_planar_LLS_Apply')
datadir = basedir.joinpath('Data_and_Figs')
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)
nbins = len(centers.iloc[:,0])
center = [12,11] # define the center of the cycle to calculate aer around
ttot = 3600
ntranslist = [1]#,2,3]
bsiter = 3000

In [3]:
### restrict data to RANDOM
treatments = ['Random']

savedir = basedir.joinpath('random')
if not savedir.exists():
    savedir.mkdir()

#restrict dataframe to only random experiments
TotalFrame = FullFrame[FullFrame.Treatment=='Random'].copy()

In [25]:
if __name__ ==  '__main__':
    ########### get raw transitions and pairs ###########
    rawtrans = DetailedBalance.get_raw_cgps_trajectories(
            TotalFrame, #pandas dataframe with all of the cgps binned data
            whichpcs, #which two PCs to use in the cgps [x,y]
            time_interval, #real time between datapoints
            savedir, #where to save the aggregated trajectories
            )


    ########### interpolate all transitions so that only individual transitions are made ###########
    transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
            rawtrans, #pandas dataframe with all of the cgps binned data
            whichpcs, #which two PCs to use in the cgps [x,y]
            savedir, #where to save the aggregated trajectories
            )
    
    
    ############## get the counts of cells leaving 
    trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
            transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
            whichpcs, #which two PCs to use in the cgps [x,y]
            savedir, #where to save the aggregated counts
            nbins, #how many bins in the x and y cgps axes
            )
    
    for ntrans in ntranslist:
        ############## BOOTSTRAP MANY TRAJECTORIES ##########
        bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                time_interval, #real time between datapoints
                savedir, #where to save the aggregated counts
                nbins, #how many bins in the x and y cgps axes
                ttot, #set the total bootstrap time
                ntrans, #how many transitions to sample at each step
                bsiter, #number of times to bootstrap
                )


        ############# calculate average bootstrapped currents ###################
        bsfield_sep = DetailedBalance.get_avg_current_error(
                bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                savedir, #where to save the aggregated counts
                nbins, #how many bins in the x and y cgps axes
                ntrans, #how many transitions to sample at each step
                )


Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 508.7579190825545 minutes
Finished finding transition rates
Boostrapping trajectories with 1 transition samples for Random


100%|██████████| 3000/3000 [03:53<00:00, 12.87it/s]


Interpolating trajectories for Random


100%|██████████| 3000/3000 [00:05<00:00, 518.64it/s] 


Calculating bootstrapped CGPS transition rates for Random


100%|██████████| 3000/3000 [02:27<00:00, 20.38it/s]


Finished bootstrapping
Boostrapping trajectories with 2 transition samples for Random


 69%|██████▉   | 2068/3000 [01:26<00:39, 23.84it/s]


KeyboardInterrupt: 

In [28]:
################ get aer and cfs ##################
if __name__ ==  '__main__':
    for ntrans in ntranslist:
        ntrans_path = savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_bootstrapped_{ntrans}_transitions.csv')
        if ntrans_path.exists():
            bstrans = pd.read_csv(ntrans_path, index_col=0)
            xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
            DetailedBalance.get_aer_cf(
                bstrans, #boostrapped transitions from get_bootstrapped_cgps_trajectories
                nbins, #how many bins in the x and y cgps axes
                xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
                center, #origin in [x bin,y bin]
                savedir, #where to save calculated aers and cfs
                whichpcs, #which two PCs to use in the cgps [x,y]
                ntrans,
                )

100%|██████████| 3000/3000 [00:13<00:00, 220.99it/s]


In [7]:
########## get RAW individual cell actual aer and cfs ###############

#get the area scaling in x and y based on the size of the bins in the cgps
xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]

#open the raw transitions in case I didn't just generate them
rawtrans = pd.read_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col = 0)

#set the origin to the actual center
results = []
for i, cells in rawtrans.groupby('CellID'):
    cells, runs = utils.get_consecutive_transitions(cells)
    for r in runs:
        cell = cells.iloc[r].reset_index(drop=True)
        results.append(DetailedBalance.get_area_enclosing_rate((
            cell,
            nbins,
            xyscaling,
            center,
            )))

#make a dataframe and save it
allaers = pd.concat(results, ignore_index=True)
justaers = allaers[['CellID','cell','Treatment','frame','real_time','time_elapsed','cumulative_time','aer','angular_velocity']].copy()
justaers.to_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))


In [10]:
############ create gaps in the bootstrapped data similar to the real data ############
ntrans = 1
justaers = pd.read_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'), index_col = 0)
realcelldf = TotalFrame.merge(justaers[['aer','cell','real_time','cumulative_time','time_elapsed']], on = 'cell', how = 'left')

########## measure gap frequency and duration
allrunlengths, allrunlengthmeans, allgaplengths, allgaplengthmeans, allgapfrequencies = DetailedBalance.get_run_stats(
        realcelldf, #dataframe
        'CellID', #what is the identifier to group by as a str
        time_interval, #frame rate of the data
        )
print(f'Average track run length mean for real data is {np.mean(allrunlengthmeans)} and mean gap frequency is {np.mean(allgapfrequencies)})')

#### get bs data with gaps
bsaers = pd.read_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_{ntrans}_transition_Area_Enclosing_Rates.csv'), index_col=0)
bstrans = pd.read_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_bootstrapped_{ntrans}_transitions.csv'), index_col = 0)

bs_with_gaps = DetailedBalance.bootstrap_runs(
    bsaers, #dataframe with bootstrap iterations (doesn't actually need aer)
    allrunlengths, #the sample of movies lengths in seconds
    allgaplengths, #the sample of non-movie gap lengths in seconds
    )

### measure the gap probability in the newly gapped bootstrap data
#change real_time to just time
bs_gap_measure = bs_with_gaps.merge(bsaers[['iter','real_time','cumulative_time','time_elapsed']], on = ['iter','real_time'], how = 'left')
#add dummy column
bs_gap_measure['aer'] = 0
bsallrunlengths, bsallrunlengthmeans, bsallgaplengths, bsallgaplengthmeans, bsallgapfrequencies = DetailedBalance.get_run_stats(
        bs_gap_measure, #dataframe
        'iter', #what is the identifier to group by as a str
        time_interval, #frame rate of the data
        )

print(f'Average track run length mean for bootstrapped data is {np.mean(bsallrunlengthmeans)} and mean gap frequency is {np.mean(bsallgapfrequencies)})')

### save the gapped bootstrap data
bs_with_gaps.to_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_{ntrans}_transition_Area_Enclosing_Rates_gaps.csv'))


Average track run length mean for real data is 27.107146663057495 and mean gap frequency is 0.006773797742590091)
Average track run length mean for bootstrapped data is 29.33992537660093 and mean gap frequency is 0.005897144906380007)


In [44]:
############ PLOT THE DISTRIBUTIONS OF REAL AND BOOTSTRAPPED GAPS AND RUNS ############
import matplotlib.pyplot as plt
import seaborn as sns



ntrans = 1
justaers = pd.read_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'), index_col = 0)
realcelldf = TotalFrame.merge(justaers[['aer','cell','real_time','cumulative_time','time_elapsed']], on = 'cell', how = 'left')

########## measure gap frequency and duration
allrunlengths, allrunlengthmeans, allgaplengths, allgaplengthmeans, allgapfrequencies = DetailedBalance.get_run_stats(
        realcelldf, #dataframe
        'CellID', #what is the identifier to group by as a str
        time_interval, #frame rate of the data
        )

#### get bs data with gaps
bsaers = pd.read_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_{ntrans}_transition_Area_Enclosing_Rates.csv'), index_col=0)
bs_with_gaps = pd.read_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_{ntrans}_transition_Area_Enclosing_Rates_gaps.csv'), index_col=0)

### measure the gap probability in the newly gapped bootstrap data
#change real_time to just time
bs_gap_measure = bs_with_gaps.merge(bsaers[['iter','real_time','cumulative_time','time_elapsed']], on = ['iter','real_time'], how = 'left')
#add dummy column
bs_gap_measure['aer'] = 0
bsallrunlengths, bsallrunlengthmeans, bsallgaplengths, bsallgaplengthmeans, bsallgapfrequencies = DetailedBalance.get_run_stats(
        bs_gap_measure, #dataframe
        'iter', #what is the identifier to group by as a str
        time_interval, #frame rate of the data
        )

############ limit bootstrapped data to long iterations only
longiters = bs_with_gaps.iter.value_counts()>realcelldf[~realcelldf.aer.isna()].CellID.value_counts().min()
longbs = bs_gap_measure[bs_gap_measure.iter.isin(longiters[longiters==True].index.to_list())]
#add dummy column
longbs['aer'] = 0
bsallrunlengthslong, bsallrunlengthmeanslong, bsallgaplengthslong, bsallgaplengthmeanslong, bsallgapfrequencieslong = DetailedBalance.get_run_stats(
        longbs, #dataframe
        'iter', #what is the identifier to group by as a str
        time_interval, #frame rate of the data
        )


################ PLOT THE REAL AND BOOTSTRAPPED DISTRIBUTIONS FOR 
################ VARIOUS RUN AND GAP STATS 
################ 
labels = ['real_data','bootstrapped_all','bootstrapped_long']


############## run lengths
data = [allrunlengthmeans,bsallrunlengthmeans,bsallrunlengthmeanslong]
all_labels = [[labels[i]]*len(x) for i, x in enumerate(data)]
tempdf = pd.DataFrame({
            'labels': np.concatenate(all_labels),
            'data': np.concatenate(data)
            })

fig, ax = plt.subplots()
sns.histplot(data = tempdf, x = 'data', hue = 'labels', bins = 60, stat = 'probability',
             element = 'step', common_norm = False, ax = ax)
ax.set_xlabel('Mean Run Length (# of frames)')
plt.savefig(savedir.joinpath('bootstrapped_data_gap_qc_mean_run_lengths.png'))



############## gap lengths
data = [allgaplengthmeans,bsallgaplengthmeans,bsallgaplengthmeanslong]
all_labels = [[labels[i]]*len(x) for i, x in enumerate(data)]
tempdf = pd.DataFrame({
            'labels': np.concatenate(all_labels),
            'data': np.concatenate(data)
            })

fig, ax = plt.subplots()
sns.histplot(data = tempdf, x = 'data', hue = 'labels', bins = 60, stat = 'probability',
             element = 'step', common_norm = False, ax = ax)
ax.set_xlabel('Mean Gap Length (# of frames)')
plt.savefig(savedir.joinpath('bootstrapped_data_gap_qc_mean_gap_lengths.png'))


############## gap frequencies
data = [allgapfrequencies,bsallgapfrequencies,bsallgapfrequencieslong]
all_labels = [[labels[i]]*len(x) for i, x in enumerate(data)]
tempdf = pd.DataFrame({
            'labels': np.concatenate(all_labels),
            'data': np.concatenate(data)
            })

fig, ax = plt.subplots()
sns.histplot(data = tempdf, x = 'data', hue = 'labels', bins = 60, stat = 'probability',
             element = 'step', common_norm = False, ax = ax)
ax.set_xlabel('Mean Gap Frequencies (# of frames)')
plt.savefig(savedir.joinpath('bootstrapped_data_gap_qc_mean_gap_frequencies.png'))

Average track run length mean for real data is 27.107146663057495 and mean gap frequency is 0.006773797742590091)


C:\ProgramData\anaconda3\envs\abhishape\lib\site-packages\ipykernel_launcher.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
